In [26]:
import torch
import torch.nn as nn

# Input Image
image = torch.randn(1, 3, 32, 32)

# Patch Embedding
patch_embed = nn.Conv2d(
    in_channels=3,
    out_channels=8,
    kernel_size=4,
    stride=4
)

patches = patch_embed(image)

# Shape:
# (1,8,8,8)

patches = patches.flatten(2).transpose(1,2)

# Shape:
# (1,64,8)

# Class Token
class_token = torch.zeros(1,1,8)

patches = torch.cat((class_token, patches), dim=1)

# Positional Encoding
position = torch.randn(1, patches.shape[1], 8)

patches = patches + position

# Layer Normalization
norm1 = nn.LayerNorm(8)

patches = norm1(patches)

# Multi-Head Self Attention
attention = nn.MultiheadAttention(
    embed_dim=8,
    num_heads=2,
    batch_first=True
)

attention_output, attention_weights = attention(
    patches,
    patches,
    patches
)

# Feed Forward Network
mlp = nn.Sequential(
    nn.Linear(8,16),
    nn.ReLU(),
    nn.Linear(16,8)
)

output = mlp(attention_output)

# Layer Normalization
norm2 = nn.LayerNorm(8)

output = norm2(output)

# Classification Head
classifier = nn.Linear(8,10)

class_output = classifier(output[:,0])

# Results
print("Patch Shape        :", patches.shape)
print("Attention Output   :", attention_output.shape)
print("Final Output Shape :", class_output.shape)
print("Attention Shape    :", attention_weights.shape)

Patch Shape        : torch.Size([1, 65, 8])
Attention Output   : torch.Size([1, 65, 8])
Final Output Shape : torch.Size([1, 10])
Attention Shape    : torch.Size([1, 65, 65])
